# Working with text data

## Tokenizing text data

In [2]:
import urllib.request 
url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
    "the-verdict.txt"
)
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x1cf71866bd0>)

In [3]:
with open(file_path, "r", encoding="utf-8") as infile:
    raw_text = infile.read()

print("Total number of characters in the text:", len(raw_text))
print(raw_text[:200])  # Print the first 200 characters of the text

Total number of characters in the text: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a


In [9]:
import re
text = "Hello, world. This is a test."
# Split on whitespaces
result = re.split(r"(\s)", text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test.']


In [10]:
text = "Hello, world. This is a test."
# Split on whitespaces, commas, and periods.
result = re.split(r"([,.]|\s)", text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [11]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', 'is', 'a', 'test', '.']


In [12]:
# NOTE: Encoding whitespaces as separate characters or just 
# removing them when developing a tokenizer depends on the 
# application. Removing whitespaces reduces memory and computing
# requirements. HOwever, keeping whitespaces can be useful if we 
# train models that are sensitive to the exact structure of the text
# (e.g., for poetry or code generation).


In [13]:
preprocessed = re.split(r'([,.:;?_!()\'"]|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


In [14]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## Converting tokens into token IDs

In [16]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print("Vocabulary size:", vocab_size)

Vocabulary size: 1130


In [18]:
vocab = {token:integer for integer,token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [19]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {integer:token for token, integer in vocab.items()}

    def encode(self, text):
        # Process input text into token IDs
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[token] for token in preprocessed]
        return ids
    
    def decode(self, ids):
        # Convert token IDs back into text
        text = " ".join([self.int_to_str[idx] for idx in ids])
        # Remove spaces before specified punctuation
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [20]:
tokenizer = SimpleTokenizerV1(vocab)
test = """It's the last he painted, you know,
        Mrs. Gisburn said with pardonable pride."""
encoded = tokenizer.encode(test)
print(encoded)

[56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 67, 7, 38, 851, 1108, 754, 793, 7]


In [21]:
tokenizer.decode(encoded)

"It' s the last he painted, you know, Mrs. Gisburn said with pardonable pride."

In [22]:
# The word Hello was not used in The Verdict, and is 
# therefore note in the vocabulary.
tokenizer.encode("Hello")

KeyError: 'Hello'

## Byte pair encoding

Byte Pair Encoding (BPE) is fundamentally a lossless compression and tokenization algorithm. It was designed for data compression by iteratively replacing the most frequent adjacent byte pairs with unused placeholder bytes, storing a lookup table to reverse the process. Because the lookup table preserves all original mappings, the original data can be perfectly reconstructed without any loss of information.

It builds a vocabulary by iteratively merging frequent characters int osubwords and frequent subwords into words. E.g., BPE starts with adding all individual characters into its vocabulary (a, b, c, etc.). Then it merges character combinations that occur frequently together into subwords. For example, d and e may be merged into de, common in many English words like define, decide, etc. The merges are determined by a frequency cutoff.  

The BPE tokenizer was used to train LLMs like GPT-2, GPT-3, and the original model used in ChatGPT.

In [24]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.13.0


In [25]:
tokenizer = tiktoken.get_encoding("gpt2")
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    " of someunknownPlace."
)
# The endoftext is a special token denoting the end of a text sequence. 
# This token signals the end of a particular text segment.
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print("Tokenized integers:\n", integers)

Tokenized integers:
 [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [26]:
strings = tokenizer.decode(integers)
print("\nDecoded string:\n", strings)


Decoded string:
 Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


In [27]:
# Try the tokenizer on unknown words
text = "Akwirw ier"

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print("Tokenized integers:\n", integers)

strings = tokenizer.decode(integers)
print("\nDecoded string:\n", strings)

Tokenized integers:
 [33901, 86, 343, 86, 220, 959]

Decoded string:
 Akwirw ier


## Data sampling with a sliding window

Generate the input-target pairs required for training an LLM.

In [30]:
with open("the-verdict.txt", "r", encoding="utf-8") as infile:
    raw_text = infile.read()

enc_text = tokenizer.encode(raw_text)

In [35]:
# Sample focusing on interesting text
enc_sample = enc_text[50:]
context_size = 4
# Input tokens 
x = enc_sample[:context_size]
# Targets = inputs shifted by 1 step
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [36]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    target = enc_sample[i]
    print(f"Context: {context} -> Target: {target}")

Context: [290] -> Target: 4920
Context: [290, 4920] -> Target: 2241
Context: [290, 4920, 2241] -> Target: 287
Context: [290, 4920, 2241, 287] -> Target: 257


In [37]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    target = enc_sample[i]
    print(f"Context: {tokenizer.decode(context)} -> Target: {tokenizer.decode([target])}")

Context:  and -> Target:  established
Context:  and established -> Target:  himself
Context:  and established himself -> Target:  in
Context:  and established himself in -> Target:  a


In [89]:
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        # Tokenizes the entire text
        token_ids = tokenizer.encode(txt)
        # Uses a sliding window to chunk the text into overlapping 
        # sequences of max_length. The stride determines how much 
        # the window moves at each step.
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        # Returns total number of rows in the dataset
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        # Returns a single row from the dataset
        return self.input_ids[idx], self.target_ids[idx]
    

def create_dataloader_v1(
    txt, 
    batch_size=4, 
    max_length=256, 
    stride=128, 
    shuffle=True,
    drop_last=True,
    num_workers=0
):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataLoader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=shuffle, 
        drop_last=drop_last, 
        num_workers=num_workers
    )
    return dataLoader


with open("the-verdict.txt", "r", encoding="utf-8") as infile:
    raw_text = infile.read()

# Stride shifts the starting point of the window in the next
# batch
dataloader = create_dataloader_v1(
    raw_text, batch_size=2, max_length=6, stride=2, shuffle=False
)
data_iter = iter(dataloader)
X, y = next(data_iter)
print("Inputs:\n", X)
print("Targets:\n", y)

Inputs:
 tensor([[  40,  367, 2885, 1464, 1807, 3619],
        [2885, 1464, 1807, 3619,  402,  271]])
Targets:
 tensor([[  367,  2885,  1464,  1807,  3619,   402],
        [ 1464,  1807,  3619,   402,   271, 10899]])


In [90]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False
)
data_iter = iter(dataloader)
X, y = next(data_iter)
print("Inputs:\n", X)
print("Targets:\n", y)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## Creating token embeddings

Convert token IDs into embedding vectors

In [92]:
# Sample data
input_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6
# Dimensionality of embedding vector
output_dim = 3

torch.manual_seed(123)
# Vocab size x embedding dimension
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [93]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [94]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


## Encoding word positions

Need a notion of position/order for the tokes within a sequence. The embedding layer maps the same token IDs to the same vector representation, regardless of the position of the token ID in the input sequence. 

In LLMs, the positional embeddings are added to the token embedding vector to create input embeddings. 

In relative positional embeddings, means the model learns the relationship in terms of distance between tokens.

In [96]:
# Same as BPE
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [98]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, 
    batch_size=8, 
    max_length=max_length, 
    stride=max_length, 
    shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("inputs shape:", inputs.shape)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
inputs shape: torch.Size([8, 4])


In [99]:
# Each token is now embedded as a 256-dimensional vector
token_embeddings = token_embedding_layer(inputs)
print("Token embeddings shape:", token_embeddings.shape)

Token embeddings shape: torch.Size([8, 4, 256])


In [100]:
# Positional embedding layer used in GPT
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
# Same embedding dimension as the token embeddings. Context length 
# is the maximum sequence length that the model can process. If the 
# input sequence is longer than the context length, we have to truncate 
# the text.
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print("Positional embeddings shape:", pos_embeddings.shape)

Positional embeddings shape: torch.Size([4, 256])


In [101]:
input_embeddings = token_embeddings + pos_embeddings
print("Input embeddings shape:", input_embeddings.shape)

Input embeddings shape: torch.Size([8, 4, 256])
